%md
# 01 — Scraper manual de grilla (Bronce)

Recorre las páginas de resultados de búsqueda (grilla) para un conjunto de
comunas y tipos de propiedad, extrayendo los datos básicos de cada aviso
publicado, como título, precio, ubicación, dormitorios, baños, superficie y
guardando solo los avisos nuevos en la tabla `avisos` de Bronce.

**Alcance de este notebook:**
- Corrida **manual**, sin scheduling, se ejecuta a mano cuando se quiera
  cargar avisos nuevos.
- Extracción vía selectores CSS y regex sobre el texto de cada tarjeta de
  resultado.
- Todo el scraping ocurre en **pandas puro** (sin la API de DataFrames de
  Spark); Spark se usa únicamente para leer los IDs ya existentes y para el
  `MERGE INTO` final hacia la tabla Delta.
- Corta la búsqueda de una comuna/tipo tras varias páginas seguidas sin
  avisos nuevos, y corta la corrida completa si se supera un techo de
  páginas o de tiempo, para no scrapear sin límite en una sola ejecución.

### 1. Verificar salida a internet
Confirma que el compute puede alcanzar el sitio antes de correr el resto del
notebook, si esto falla, no tiene sentido seguir.

In [0]:
import requests
resp = requests.get("https://www.portalinmobiliario.com", timeout=15)

if resp.status_code == 200:
    print("Conección exitosa!")
else:
    print("Error!")


### 2. Instalar dependencias
BeautifulSoup y lxml para parsear el HTML de cada página de resultados.

In [0]:
import re
import time
import random
import logging
from datetime import date
from dataclasses import dataclass, asdict
from typing import Optional

import requests
from bs4 import BeautifulSoup
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

COMUNAS_GRAN_CONCEPCION = [
    "concepcion-biobio",
    "talcahuano-biobio",
    "hualpen-biobio",
    "san-pedro-de-la-paz-biobio",
    "chiguayante-biobio",
    "penco-biobio",
    "tome-biobio",
    "coronel-biobio",
    "hualqui-biobio",
    "lota-biobio",
]

# Solo departamento, igual que el scraper de producción (el resto del
# pipeline no procesa casas)
TIPOS_PROPIEDAD_PRODUCCION = ["departamento"]
OPERACION = "arriendo"

MAX_PAGINAS_POR_BUSQUEDA = 1000
MAX_PAGINAS_VACIAS_CONSECUTIVAS = 10
MAX_PAGINAS_POR_CORRIDA = 200
MAX_MINUTOS_POR_CORRIDA = 30
RESULTADOS_POR_PAGINA = 48
DELAY_MIN = 3.0
DELAY_MAX = 7.0

BASE_URL = "https://www.portalinmobiliario.com"

RE_ID_AVISO = re.compile(r"(MLC-\d+)")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}

SELECTORES = {
    "tarjeta": ["div.ui-search-result__wrapper", "div.andes-card", "li.ui-search-layout__item"],
    "titulo": ["h2.ui-search-item__title", "h3.poly-component__title", "a.poly-component__title"],
    "link": ["a.ui-search-link", "a.poly-component__title"],
    "precio": ["span.andes-money-amount__fraction"],
    "moneda": ["span.andes-money-amount__currency-symbol"],
    "ubicacion": ["span.ui-search-item__location", "span.poly-component__location"],
}

RE_DORMITORIOS = re.compile(r"(\d+)\s*dormitorios?", re.IGNORECASE)
RE_BANOS = re.compile(r"(\d+)\s*ba[ñn]os?", re.IGNORECASE)
RE_M2 = re.compile(r"([\d.,]+)\s*m[²2]\b", re.IGNORECASE)


@dataclass
class Aviso:
    comuna: str
    tipo_propiedad: str
    operacion: str
    id_aviso: Optional[str] = None
    titulo: Optional[str] = None
    precio: Optional[str] = None
    moneda: Optional[str] = None
    ubicacion: Optional[str] = None
    dormitorios: Optional[str] = None
    banos: Optional[str] = None
    superficie_m2: Optional[str] = None
    url: Optional[str] = None

### 3. Crear el esquema de Bronce y la tabla `avisos` (si no existen)
Para que el pipeline se pueda reconstruir desde cero sin pasos manuales: si
el catálogo está vacío, este bloque deja creado el esquema `01_bronce`, la
tabla `avisos` con su esquema completo (incluidas las columnas de estado de
publicación que usa el re-chequeo del notebook 02) y la tabla `control`
(clave/valor genérica para estado interno de los scrapers). Si ya existen,
`IF NOT EXISTS` no hace nada.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gran_concepcion.01_bronce")

spark.sql("""
    CREATE TABLE IF NOT EXISTS gran_concepcion.01_bronce.avisos (
        id_aviso                     STRING NOT NULL,
        comuna                       STRING,
        tipo_propiedad               STRING,
        operacion                    STRING,
        titulo                       STRING,
        precio                       STRING,
        moneda                       STRING,
        ubicacion                    STRING,
        dormitorios                  STRING,
        banos                        STRING,
        superficie_m2                STRING,
        url                          STRING,
        first_seen                   STRING,
        intentos_fallidos_detalle    INT,
        estado_publicacion           STRING,
        fecha_ultimo_chequeo_estado  DATE
    )
""")

# Tabla clave/valor genérica para estado interno de los scrapers (ej. cooldown
# tras CAPTCHA del scraper de detalle, ver 02_scraper_manual_detalle_bronce).
# Se crea acá porque este notebook corre primero en la secuencia manual.
spark.sql("""
    CREATE TABLE IF NOT EXISTS gran_concepcion.01_bronce.control (
        clave STRING NOT NULL,
        valor STRING
    )
""")

print("Esquema y tablas de Bronce verificados/creados.")


### 5. Funciones de scraping y parsing
Construcción de URLs de búsqueda, descarga de HTML, y extracción de los
campos de cada tarjeta (título, precio, ubicación, dormitorios, baños,
superficie) desde el HTML de una página de resultados.

In [0]:
def extraer_id_aviso(url):
    if not url:
        return None
    m = RE_ID_AVISO.search(url)
    return m.group(1) if m else None


def _first_match(soup_or_tag, selector_list):
    for sel in selector_list:
        found = soup_or_tag.select(sel)
        if found:
            return found
    return []


def construir_url(tipo_propiedad, comuna, pagina):
    offset = 1 + (pagina - 1) * RESULTADOS_POR_PAGINA
    if pagina == 1:
        return f"{BASE_URL}/{OPERACION}/{tipo_propiedad}/{comuna}"
    return f"{BASE_URL}/{OPERACION}/{tipo_propiedad}/{comuna}/_Desde_{offset}_NoIndex_True"


def obtener_html(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
    except requests.RequestException as e:
        log.warning(f"Error de red en {url}: {e}")
        return None

    if resp.status_code != 200:
        log.warning(f"Status {resp.status_code} en {url}")
        return None

    if "captcha" in resp.text.lower()[:5000]:
        log.warning(f"Posible CAPTCHA detectado en {url}. Deteniendo esta búsqueda.")
        return None

    return resp.text


def extraer_atributo_texto(tag, selector_list):
    encontrados = _first_match(tag, selector_list)
    if encontrados:
        return encontrados[0].get_text(strip=True)
    return None


def parsear_atributos_regex(tarjeta):
    texto_completo = tarjeta.get_text(" ", strip=True)
    resultado = {"dormitorios": None, "banos": None, "superficie_m2": None}

    m = RE_DORMITORIOS.search(texto_completo)
    if m:
        resultado["dormitorios"] = m.group(1)
    m = RE_BANOS.search(texto_completo)
    if m:
        resultado["banos"] = m.group(1)
    m = RE_M2.search(texto_completo)
    if m:
        resultado["superficie_m2"] = m.group(1)

    return resultado


def parsear_pagina(html, comuna, tipo_propiedad):
    soup = BeautifulSoup(html, "lxml")
    tarjetas = _first_match(soup, SELECTORES["tarjeta"])

    if not tarjetas:
        log.info(f"Sin tarjetas encontradas ({comuna}, {tipo_propiedad}).")
        return []

    avisos = []
    for tarjeta in tarjetas:
        titulo = extraer_atributo_texto(tarjeta, SELECTORES["titulo"])
        precio = extraer_atributo_texto(tarjeta, SELECTORES["precio"])
        moneda = extraer_atributo_texto(tarjeta, SELECTORES["moneda"])
        ubicacion = extraer_atributo_texto(tarjeta, SELECTORES["ubicacion"])

        link_tag = _first_match(tarjeta, SELECTORES["link"])
        url = link_tag[0].get("href") if link_tag else None
        atributos = parsear_atributos_regex(tarjeta)

        avisos.append(Aviso(
            comuna=comuna,
            tipo_propiedad=tipo_propiedad,
            operacion=OPERACION,
            id_aviso=extraer_id_aviso(url),
            titulo=titulo,
            precio=precio,
            moneda=moneda,
            ubicacion=ubicacion,
            dormitorios=atributos["dormitorios"],
            banos=atributos["banos"],
            superficie_m2=atributos["superficie_m2"],
            url=url,
        ))
    return avisos

### 6. Cargar IDs de avisos ya existentes
Trae los IDs ya guardados en Bronce, para no volver a insertar avisos que
ya se scrapearon en una corrida anterior.

In [0]:
ids_conocidos = set(
    row["id_aviso"] for row in
    spark.sql("SELECT id_aviso FROM gran_concepcion.01_bronce.avisos").collect()
)
log.info(f"{len(ids_conocidos)} avisos ya existen en Bronce.")

### 7. Scraping de la grilla, en memoria
Recorre cada combinación de comuna y tipo de propiedad, página por página,
guardando en memoria solo los avisos cuyo ID todavía no existe. Corta por
páginas vacías consecutivas, por techo de páginas totales, o por techo de
tiempo, lo que ocurra primero.

In [ ]:
avisos_nuevos_dicts = []
total_vistos = 0
total_nuevos = 0
paginas_recorridas = 0
motivo_corte = None
t0 = time.time()
hoy = date.today().isoformat()

for comuna in COMUNAS_GRAN_CONCEPCION:
    if motivo_corte in ("limite_paginas", "limite_tiempo"):
        break

    for tipo in TIPOS_PROPIEDAD_PRODUCCION:
        if motivo_corte in ("limite_paginas", "limite_tiempo"):
            break

        log.info(f"--- Buscando: {OPERACION} de {tipo} en {comuna} ---")
        paginas_vacias_consecutivas = 0
        pagina = 1

        while True:
            if paginas_recorridas >= MAX_PAGINAS_POR_CORRIDA:
                motivo_corte = "limite_paginas"
                break
            if (time.time() - t0) / 60 >= MAX_MINUTOS_POR_CORRIDA:
                motivo_corte = "limite_tiempo"
                break
            if paginas_vacias_consecutivas >= MAX_PAGINAS_VACIAS_CONSECUTIVAS:
                break
            if pagina > MAX_PAGINAS_POR_BUSQUEDA:
                break

            url = construir_url(tipo, comuna, pagina)
            log.info(f"Página {pagina}: {url}")
            html = obtener_html(url)
            paginas_recorridas += 1

            if html is None:
                break

            avisos = parsear_pagina(html, comuna, tipo)
            if not avisos:
                break

            nuevos_en_pagina = 0
            for aviso in avisos:
                total_vistos += 1
                if not aviso.id_aviso or aviso.id_aviso in ids_conocidos:
                    continue

                d = asdict(aviso)
                d["first_seen"] = hoy
                d["intentos_fallidos_detalle"] = 0

                avisos_nuevos_dicts.append(d)
                ids_conocidos.add(aviso.id_aviso)
                nuevos_en_pagina += 1
                total_nuevos += 1

            paginas_vacias_consecutivas = 0 if nuevos_en_pagina > 0 else paginas_vacias_consecutivas + 1
            log.info(f"  -> {len(avisos)} vistos, {nuevos_en_pagina} nuevos "
                      f"(vacías: {paginas_vacias_consecutivas}/{MAX_PAGINAS_VACIAS_CONSECUTIVAS})")

            pagina += 1
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

if motivo_corte is None:
    motivo_corte = "paginas_vacias_consecutivas"

log.info(
    f"Corrida completa. Vistos: {total_vistos} | Nuevos: {total_nuevos} | "
    f"Páginas: {paginas_recorridas} | Motivo de corte: {motivo_corte} | "
    f"Duración: {round(time.time() - t0, 1)}s"
)

### 8. Armar DataFrame con los avisos nuevos
Convierte lo recolectado en memoria a un DataFrame de pandas, listo para
insertar.

In [0]:
df_nuevos = pd.DataFrame(avisos_nuevos_dicts)
print(f"{len(df_nuevos)} avisos nuevos para insertar")
df_nuevos.head()

### 9. Crear vista temporal para el INSERT
Punto único de contacto con Spark: convierte el DataFrame de pandas en una
vista SQL temporal.

In [0]:
spark.createDataFrame(df_nuevos).createOrReplaceTempView("avisos_nuevos_tmp")

### 10. MERGE final hacia Bronce
Inserta en `avisos` solo los avisos que todavía no existían (columnas
explícitas + casts, para evitar pérdidas silenciosas de tipo).

In [ ]:
%sql
MERGE INTO gran_concepcion.01_bronce.avisos AS avisos
USING avisos_nuevos_tmp AS nuevos
ON avisos.id_aviso = nuevos.id_aviso
WHEN NOT MATCHED THEN INSERT (
    id_aviso, comuna, tipo_propiedad, operacion, titulo, precio,
    moneda, ubicacion, dormitorios, banos, superficie_m2, url,
    first_seen, intentos_fallidos_detalle, estado_publicacion
) VALUES (
    nuevos.id_aviso, 
    nuevos.comuna, 
    nuevos.tipo_propiedad, 
    nuevos.operacion,
    nuevos.titulo, 
    nuevos.precio, 
    nuevos.moneda, 
    nuevos.ubicacion,
    nuevos.dormitorios, 
    nuevos.banos, 
    nuevos.superficie_m2, 
    nuevos.url,
    nuevos.first_seen, 
    nuevos.intentos_fallidos_detalle,
    'activo'
)